In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Patch


In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")


In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
# Read the *shapefile* explicitly (not the folder)
osm_places_path = base_path / "dphil_common_cross_cutting/common_incoming_data/osm/gis_osm_places_free_1/gis_osm_places_free_1.shp"
osm_places = gpd.read_file(osm_places_path)
print("OSM places CRS as read:", osm_places.crs)

# --- Sanity check: are the coords actually degrees? ---
x_min, x_max = float(osm_places.geometry.x.min()), float(osm_places.geometry.x.max())
print(f"x-range: {x_min:.3f} .. {x_max:.3f}")


In [ ]:
osm_places_3448 = osm_places.to_crs(jamaica_boundary.crs)
osm_places_3448.crs

In [ ]:
# 2) Build a single mask polygon (nice-to-have for a clean clip)
jamaica_mask = jamaica_boundary.dissolve().reset_index(drop=True)

# 3) Clip places to Jamaica
osm_places_jamaica_3448 = gpd.clip(osm_places_3448, jamaica_mask)

print("CRS:", osm_places_jamaica_3448.crs)
print(f"Places total: {len(osm_places):,}  |  in Jamaica: {len(osm_places_jamaica_3448):,}")

osm_places_jamaica_3448.head()

In [ ]:
# 4) (Optional) keep only major settlements (OSM fclass 'city'/'town')
major_places_jamaica_3448 = osm_places_jamaica_3448[osm_places_jamaica_3448["fclass"].isin(["city", "town"])].copy()
print(f"Major (city/town): {len(major_places_jamaica_3448):,}")


In [ ]:
catchments = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal.gpkg"
catchments = gpd.read_file(catchments)

In [ ]:
rivers_path = base_path / "dphil_common_cross_cutting/common_incoming_data/osm/gis_osm_waterways_free_1/gis_osm_waterways_free_1.shp"
rivers = gpd.read_file(rivers_path)

# simple_rivers_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/NbS_river_catchment/rivers.gpkg")
# rivers_simple = gpd.read_file(simple_rivers_path)
# rivers_simple.crs
# rivers_3448 = rivers_simple


rivers.crs
target_crs = "EPSG:3448"
rivers_3448 = rivers.to_crs(target_crs)
rivers_3448.crs
rivers_3448.head()

In [ ]:
CATCH_NAME_COL = next((c for c in ["name","NAME","basin","BASIN","Catchment","CATCHMENT"] if c in catchments.columns), catchments.columns[0])


In [ ]:
# Prefer OSM 'name'; if it's missing/blank, fall back to the row index as a string
if "name" in rivers_3448.columns and rivers_3448["name"].notna().any():
    rivers_3448["river_id"] = rivers_3448["name"].where(
        rivers_3448["name"].notna() & (rivers_3448["name"].astype(str).str.strip() != ""),
        rivers_3448.index.to_series().astype(str)   # <-- Series, not Index
    )
elif "osm_id" in rivers_3448.columns:
    rivers_3448["river_id"] = rivers_3448["osm_id"].astype("string").fillna(
        rivers_3448.index.to_series().astype(str)
    )
else:
    rivers_3448["river_id"] = rivers_3448.index.astype(str)

In [ ]:
# Clip rivers by catchments (line parts inside each polygon)
parts = gpd.overlay(
    rivers_3448[["river_id", "geometry"]],
    catchments[[CATCH_NAME_COL, "geometry"]],
    how="intersection"
).rename(columns={CATCH_NAME_COL: "catchment_name"})

# Length of each clipped segment
parts["part_len_m"] = parts.geometry.length

# Sum length per (river, catchment)
lens = (
    parts.groupby(["river_id", "catchment_name"], dropna=False)["part_len_m"]
         .sum().reset_index()
)



In [ ]:
# Pick the catchment with max length for each river
idx = lens.groupby("river_id")["part_len_m"].idxmax()
majority = lens.loc[idx].reset_index(drop=True)  # columns: river_id, catchment_name, part_len_m

# Total length per river (for share)
river_len = rivers_3448.assign(total_len_m=rivers_3448.geometry.length)[["river_id", "total_len_m"]]

majority = majority.merge(river_len, on="river_id", how="left")
majority["share_in_catchment"] = np.where(majority["total_len_m"] > 0,
                                          majority["part_len_m"] / majority["total_len_m"], np.nan)

# Join back to the original rivers
rivers_with_catch = rivers_3448.merge(
    majority[["river_id", "catchment_name", "share_in_catchment"]],
    on="river_id", how="left"
)



In [ ]:
# ... you already built rivers_with_catch above ...

missing = rivers_with_catch["catchment_name"].isna()
if missing.any():
    catch_3448 = catchments.to_crs(rivers_3448.crs)  # ensure same CRS
    nearest = gpd.sjoin_nearest(
        rivers_with_catch.loc[missing, ["geometry"]],
        catch_3448[[CATCH_NAME_COL, "geometry"]],
        how="left",
        distance_col="dist_m"
    ).rename(columns={CATCH_NAME_COL: "catchment_name_near"})

    # Collapse any ties (multiple nearest) to a single match per left feature
    # Important: sjoin_nearest keeps the LEFT index as the result index
    nearest = nearest[["catchment_name_near"]].groupby(level=0).first()

    # Align by index to exactly the missing rows (same length, no mismatch)
    fill = nearest.reindex(rivers_with_catch.index[missing])["catchment_name_near"]
    rivers_with_catch.loc[missing, "catchment_name"] = fill.values

    # share unknown for nearest-based fill
    rivers_with_catch.loc[missing, "share_in_catchment"] = np.nan

# Build catchments layer with river info attached

In [ ]:
catch_3448.head()

In [ ]:
# 1) Choose the catchment name column you found earlier
# catch_3448 = catchments.reset_index(drop=False).rename(columns={"index": "catch_id"})
# catch_3448["catchment_name"] = catch_3448[CATCH_NAME_COL].astype(str)



# Use existing uid column instead of index-based catch_id
catch_3448 = catchments.copy()

# If your column is literally named "uid"
catchment_uid_col = "uid"

catch_3448["catchment_name"] = catch_3448[CATCH_NAME_COL].astype(str)


# 2) Prepare river identifiers / labels
#    Use a stable id (osm_id if present, else index) and a human-readable label (name if present)
rivers_3448["river_id"] = (
    (rivers_3448["osm_id"].astype(str) if "osm_id" in rivers_3448.columns else rivers_3448.index.astype(str))
)
rivers_3448["river_name"] = (
    rivers_3448["name"].astype("string").str.strip() if "name" in rivers_3448.columns else pd.Series("", index=rivers_3448.index, dtype="string")
)
rivers_3448["river_label"] = rivers_3448["river_name"].where(rivers_3448["river_name"].notna() & (rivers_3448["river_name"] != ""),
                                                             "Unnamed " + rivers_3448["river_id"])

# 3) Clip rivers by catchments (line parts inside polygons)
# parts = gpd.overlay(
#     rivers_3448[["river_id", "river_label", "geometry"]],
#     catch_3448[["catch_id", "catchment_name", "geometry"]],
#     how="intersection"
# )
# parts["len_m"] = parts.geometry.length


parts = gpd.overlay(
    rivers_3448[["river_id", "river_label", "geometry"]],
    catch_3448[[catchment_uid_col, "catchment_name", "geometry"]],
    how="intersection"
)
parts["len_m"] = parts.geometry.length


# 4) Sum length per (catchment, river)
len_tbl = (
    parts.groupby(["catch_id", "catchment_name", "river_id", "river_label"], dropna=False)["len_m"]
         .sum()
         .reset_index()
)

# 5) Primary river per catchment (longest inside)
primary = (
    len_tbl.sort_values(["catch_id", "len_m"], ascending=[True, False])
           .groupby("catch_id", as_index=False)
           .first()
           .rename(columns={"river_label": "primary_river", "len_m": "primary_river_len_m"})
)[["catch_id", "primary_river", "primary_river_len_m"]]

# 6) List of all rivers per catchment (unique, order of first appearance)
def uniq_join(series):
    seen = set()
    out = []
    for v in series:
        if pd.isna(v): 
            continue
        if v not in seen:
            seen.add(v)
            out.append(v)
    return ", ".join(out) if out else None

river_list = (
    len_tbl.groupby("catch_id")["river_label"]
           .apply(uniq_join)
           .reset_index()
           .rename(columns={"river_label": "rivers_list"})
)

# 7) Attach to catchments
catchments_with_river_info = (
    catch_3448
    .merge(primary,   on="catch_id", how="left")
    .merge(river_list, on="catch_id", how="left")
)

# 8) (Optional) Save
# out_fp = base_path / "Outputs" / "catchments_with_river_info.gpkg"
# catchments_with_river_info.to_file(out_fp, driver="GPKG")

In [ ]:
catchments_with_river_info

In [ ]:

# ============== options ==============
SHADE_BY_RIVER = True   # color polygons by primary_river
SHOW_LABELS    = True   # write primary_river name inside each catchment
TITLE          = "Catchments by primary (longest) river"
# =====================================

# 0) Ensure projected CRS for lengths/label points (use same as your workflow)
target_crs = "EPSG:3448"
gdf = catchments_with_river_info.to_crs(target_crs).copy()

# (Optional) rivers for context if you have them in memory; otherwise this will be skipped
rivers_plot = None
try:
    # if you have 'rivers' or 'rivers_3448' available:
    if 'rivers_3448' in globals():
        rivers_plot = rivers_3448.to_crs(target_crs)
    elif 'rivers' in globals():
        rivers_plot = rivers.to_crs(target_crs)
except Exception:
    rivers_plot = None

# 1) Clean up the label column
label_col = "primary_river"
if label_col not in gdf.columns:
    raise ValueError(f"Column '{label_col}' not found in catchments_with_river_info.")

gdf[label_col] = gdf[label_col].astype("string")
# Optional: replace totally empty with a friendlier label
gdf[label_col] = gdf[label_col].where(gdf[label_col].str.len() > 0, other=None)

# 2) Colors for categories (primary_river)
if SHADE_BY_RIVER:
    cats = gdf[label_col].fillna("No primary river")
    codes, uniques = pd.factorize(cats, sort=True)
    # Use a qualitative colormap sized to unique rivers
    cmap = plt.get_cmap("tab20", max(1, len(uniques)))
    gdf["_color"] = [cmap(i) for i in codes]
else:
    gdf["_color"] = "#E6E6E6"  # neutral fill if not shading

# 3) Plot
fig, ax = plt.subplots(figsize=(10, 8))

# polygons
gdf.plot(ax=ax, color=gdf["_color"], edgecolor="white", linewidth=0.6, zorder=1)

# rivers (context)
if rivers_plot is not None and not rivers_plot.empty:
    rivers_plot.plot(ax=ax, color="none", edgecolor="#4AA3DF", linewidth=0.6, alpha=0.7, zorder=2)

# labels
if SHOW_LABELS:
    # representative_point() places point guaranteed inside polygon (better than centroid for concave shapes)
    pts = gdf.representative_point()
    for (x, y), name in zip(pts.geometry.apply(lambda p: (p.x, p.y)), gdf[label_col]):
        if pd.isna(name) or name == "No primary river":
            continue
        ax.text(
            x, y, str(name),
            ha="center", va="center", fontsize=8, color="#111",
            path_effects=[pe.withStroke(linewidth=3, foreground="white")],
            zorder=3
        )

# legend (only if shading by river and a small-ish number of categories)
if SHADE_BY_RIVER:
    # If many unique rivers, a legend can get huge—show only if reasonable
    uniq_vals = gdf[[label_col, "_color"]].drop_duplicates().sort_values(label_col, na_position="last")
    if len(uniq_vals) <= 15:  # tweak threshold to taste
        handles = [
            Patch(facecolor=row["_color"], edgecolor="white", label=("No primary river" if pd.isna(row[label_col]) else row[label_col]))
            for _, row in uniq_vals.iterrows()
        ]
        leg = ax.legend(handles=handles, title="Primary river", frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1))
        fig.subplots_adjust(right=0.82)

ax.set_title(TITLE, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()




# ---- Save (to your existing paths) ----
# from pathlib import Path
# out_dir.mkdir(parents=True, exist_ok=True)
# fname = out_dir / "catchments_by_primary_river"
# fig.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
# fig.savefig(fname.with_suffix(".pdf"),              bbox_inches="tight")

plt.show()
# print("Saved to:", fname.with_suffix(".png"), "and", fname.with_suffix(".pdf"))

In [ ]:
g = (catchments_with_river_info
     .dropna(subset=["primary_river"])
     .groupby("primary_river")
     .agg(
         n_catch=("catchment_name", "count"),
         catchments=("catchment_name", lambda s: ", ".join(s))
     )
     .query("n_catch > 1")
     .sort_values("n_catch", ascending=False))

print(g)

In [ ]:
gdf.to_file("catchments_with_river_v1.gpkg")